Links necessários (caso os datasets não estejam disponíveis no repositório): 
- https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar(Censo Escolar de 2024)
- https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/indicadores-educacionais/taxas-de-rendimento-escolar (Rendimento Escolar por Escola em 2024)

<h1>
    Criação do ambiente conda para execução
</h1>

```
conda create -n abandono_spark python=3.12 pyspark pandas openpyxl ipykernel -y
conda activate abandono_spark
conda install -c conda-forge openjdk=17 -y
```

Instalação dos pacotes necessários:

In [1]:
%pip install --only-binary=:all: pyspark pandas openpyxl #Talvez não precise da flag only-binary, mas é bom para evitar problemas de compilação em algumas máquinas.

Note: you may need to restart the kernel to use updated packages.


## Definindo paths

In [6]:
microdados_path = "data/microdados_censo_escolar_2024_defeso/dados/microdados_ed_basica_2024.csv"
tx_rendimento_path = "data/tx_rend_escolas_2024_convertido.csv"
pre_tx_rendimento_path = "data/tx_rend_escolas_2024.xlsx"

<h2>Conversão dos arquivos XLSX/ODS para CSV<h2>

In [7]:
import pandas as pd
import os

print("Iniciando conversão do Excel para CSV...")

# 1. Lendo o Excel usando Pandas. 
# header = 7 (cabeçalho)
df_pandas = pd.read_excel(pre_tx_rendimento_path, sheet_name='ESCOLAS', header=7)

# 2. Salvando como CSV (separado por ponto e vírgula, padrão no Brasil)
df_pandas.to_csv(tx_rendimento_path, sep=';', index=False, encoding='utf-8')

print(f"Conversão concluída! Arquivo salvo como: {tx_rendimento_path}")

Iniciando conversão do Excel para CSV...
Conversão concluída! Arquivo salvo como: data/tx_rend_escolas_2024_convertido.csv


<h2> Iniciando o Spark </h2>

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, regexp_replace

# Criando a sessão do Spark com paralelismo total
spark = SparkSession.builder \
    .appName("Pipeline_Evasao_Escolar") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Sessão Spark iniciada com sucesso!")
print("Versão do Spark:", spark.version)

Sessão Spark iniciada com sucesso!
Versão do Spark: 4.1.1


In [6]:
# 1. Ingestão da base de Infraestrutura (Censo Escolar Original de 250MB)
df_infra = spark.read.csv(microdados_path, header=True, sep=";")

# 2. Ingestão da base de Rendimento (O CSV que acabamos de converter)
df_rend = spark.read.csv(tx_rendimento_path, header=True, sep=";")

print("--- Colunas de Infraestrutura ---")
df_infra.printSchema()

print("--- Colunas de Rendimento ---")
df_rend.printSchema()

--- Colunas de Infraestrutura ---
root
 |-- NU_ANO_CENSO: string (nullable = true)
 |-- NO_REGIAO: string (nullable = true)
 |-- CO_REGIAO: string (nullable = true)
 |-- NO_UF: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- CO_UF: string (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- CO_MUNICIPIO: string (nullable = true)
 |-- NO_REGIAO_GEOG_INTERM: string (nullable = true)
 |-- CO_REGIAO_GEOG_INTERM: string (nullable = true)
 |-- NO_REGIAO_GEOG_IMED: string (nullable = true)
 |-- CO_REGIAO_GEOG_IMED: string (nullable = true)
 |-- NO_MESORREGIAO: string (nullable = true)
 |-- CO_MESORREGIAO: string (nullable = true)
 |-- NO_MICRORREGIAO: string (nullable = true)
 |-- CO_MICRORREGIAO: string (nullable = true)
 |-- NO_DISTRITO: string (nullable = true)
 |-- CO_DISTRITO: string (nullable = true)
 |-- NO_ENTIDADE: string (nullable = true)
 |-- CO_ENTIDADE: string (nullable = true)
 |-- TP_DEPENDENCIA: string (nullable = true)
 |-- TP_CATEGORIA_ESCOLA

<h2> Fazendo a ingestão dos dados </h2>

In [5]:
# 1. Ingestão da base de Infraestrutura (Censo Escolar Original de 250MB)
df_infra = spark.read.csv(microdados_path, header=True, sep=";")

# 2. Ingestão da base de Rendimento (O CSV que acabamos de converter)
df_rend = spark.read.csv(tx_rendimento_path, header=True, sep=";")

# Limpeza inicial: garantir que o código da escola (CO_ENTIDADE) contenha apenas números
# O INEP costuma ter linhas de rodapé soltas no arquivo.
df_infra = df_infra.filter(col("CO_ENTIDADE").rlike("^[0-9]+$"))
df_rend = df_rend.filter(col("CO_ENTIDADE").rlike("^[0-9]+$"))

print(f"Linhas em Infraestrutura: {df_infra.count()}")
print(f"Linhas em Rendimento: {df_rend.count()}")

26/07/09 11:12:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
{"ts": "2026-07-09 11:12:04.823", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `CO_ENTIDADE` cannot be resolved. Did you mean one of the following? [`Total`, ` 5º Ano`, ` 5º Ano.1`, ` 5º Ano.2`, `1ª série`]. SQLSTATE: 42703", "context": {"file": "line 10 in cell [6]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o42.filter.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `CO_ENTIDADE` cannot be resolved. Did you mean one of the following? [`Total`, ` 5º Ano`, ` 5º Ano.1`, ` 5º Ano.2`, `1ª série`

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `CO_ENTIDADE` cannot be resolved. Did you mean one of the following? [`Total`, ` 5º Ano`, ` 5º Ano.1`, ` 5º Ano.2`, `1ª série`]. SQLSTATE: 42703;
'Filter 'rlike('CO_ENTIDADE, ^[0-9]+$)
+- Relation [Unnamed: 0#460,Unnamed: 1#461,Unnamed: 2#462,Unnamed: 3#463,Unnamed: 4#464,Unnamed: 5#465,Unnamed: 6#466,Unnamed: 7#467,Unnamed: 8#468,Total#469,Anos Iniciais#470,Anos Finais#471,1º Ano #472,2º Ano#473,3º Ano#474,4º Ano#475, 5º Ano#476,6º Ano#477,7º Ano#478,8º Ano#479,9º Ano#480,Total  #481,1ª série#482,2ª série#483,3ª série#484,... 38 more fields] csv
